# Wizualizacja Danych Live - Kompletny Przegląd

**Cel:** Zrozumieć jak wizualizować dane które **zmieniają się w czasie rzeczywistym** i jak to opublikować.

---

## 🎯 Czym są "dane na żywo"?

**Dane live (streaming data)** to dane które:
- Napływają **ciągle** (nie mamy całego zbioru od razu)
- Zmieniają się w **czasie rzeczywistym**
- Wymagają **bieżącej aktualizacji** wizualizacji

### Przykłady:
- 📊 Dane giełdowe (ceny akcji co sekundę)
- 🌡️ Czujniki IoT (temperatura, wilgotność)
- 📈 Metryki serwerów (CPU, RAM, requests/sec)
- 🚗 Lokalizacja GPS pojazdów
- 💬 Social media feeds (tweety, posty)
- 🏭 Dane produkcyjne z maszyn

---

## ⚠️ WAŻNE: "Fake Live" vs "Real Live"

Istnieją **różne poziomy** "live" - od prostych symulacji po prawdziwy streaming.

---

## Scenariusz 1: "Fake Live" - Symulacja (co robiliśmy)

### Jak działa:
```python
# Dash z dcc.Interval
dcc.Interval(interval=1000)  # Co 1 sekundę

@app.callback(...)
def update(n):
    new_value = random.randint(10, 100)  # <-- FAKE! Losowe dane
    return wykres
```

### Charakterystyka:
✅ **Plusy:**
- Bardzo proste do zrobienia
- Nie potrzeba żadnej infrastruktury
- Świetne do prototypów i nauki
- Działa lokalnie bez external services

❌ **Minusy:**
- Dane są **sztuczne** (generowane)
- Brak prawdziwego źródła danych
- Nie skaluje się (każdy użytkownik = osobny generator)

### Kiedy używać:
- **Prototypy** interfejsu
- **Demo** dla klienta (pokazać jak będzie wyglądać)
- **Nauka** Dash i mechanizmów callback
- **Testy** layoutu przed podłączeniem prawdziwych danych

**To NIE jest rozwiązanie produkcyjne!**

---

## Scenariusz 2: Polling Bazy Danych - "Semi-Live"

### Jak działa:
```python
# Dash z dcc.Interval + baza danych
dcc.Interval(interval=5000)  # Co 5 sekund

@app.callback(...)
def update(n):
    # Czytaj najnowsze dane z bazy
    conn = sqlite3.connect('sensors.db')
    df = pd.read_sql('SELECT * FROM readings ORDER BY timestamp DESC LIMIT 100', conn)
    return wykres_z_df
```

### Charakterystyka:
✅ **Plusy:**
- **Prawdziwe dane** z bazy
- Relatywnie proste (Dash + SQL)
- Dane są trwałe (zapisane w DB)
- Wielu użytkowników widzi te same dane

❌ **Minusy:**
- **Opóźnienie** (polling co X sekund)
- Obciążenie bazy danych (każdy interval = query)
- Nie jest "true real-time" (mogą być luki)
- Marnuje zasoby (pytamy nawet gdy brak zmian)

### Kiedy używać:
- Dane aktualizują się **rzadko** (co minutę, co 5 minut)
- Opóźnienie 5-60 sekund jest **akceptowalne**
- Masz już bazę danych ze zebranymi danymi
- Dashboard dla **małej grupy** użytkowników (do ~50 osób)

### Przykłady:
- Dashboard sprzedaży (aktualizacja co 5 min)
- Monitoring serwerowni (co 30 sec)
- Dashboard pogodowy (co 10 min)

**To najczęstsze rozwiązanie dla małych/średnich projektów!**

---

## Scenariusz 3: Prawdziwy Streaming - "True Real-Time"

### Jak działa:
Dane **płyną** do dashboardu w momencie powstania (milisekundy opóźnienia).

**Architektura:**
```
Źródło danych → Streaming Platform → Dashboard
  (sensory)         (Kafka/WS)        (Dash)
```

### Charakterystyka:
✅ **Plusy:**
- **Prawdziwy real-time** (milisekundy opóźnienia)
- Skalowalne (miliony eventów/sekundę)
- Efektywne (push, nie polling)
- Dane są "strumieniowane" nie "ciągnięte"

❌ **Minusy:**
- **Złożona infrastruktura** (Kafka, message brokers)
- Wymaga dedykowanych narzędzi
- Trudniejsze deployment i utrzymanie
- Kosztowniejsze (więcej serwisów)

### Kiedy używać:
- Opóźnienie **musi być minimalne** (< 1 sekundy)
- **Ogromne wolumeny** danych (tysiące eventów/sec)
- Aplikacje **mission-critical** (finanse, monitoring infrastruktury)
- Wiele systemów **współdzieli** te same dane

### Przykłady:
- Trading platforms (giełda)
- Monitoring systemów critical (elektrownie, lotniska)
- IoT w wielkim stylu (tysiące czujników)
- Live analytics (clickstream na stronie)

**To rozwiązanie enterprise - duże projekty, duże budżety.**

---

## 🛠️ Ekosystem Narzędzi - Przegląd Technologii

Poniżej **najpopularniejsze narzędzia** do streamingu danych. Nie musisz ich wszystkich znać, ale warto wiedzieć że istnieją!

### 1. Apache Kafka - "Król Streamingu"

**Co to jest:**
- Rozproszona platforma do **streamingu danych**
- Działa jak "highway" dla danych - producenci wysyłają, konsumenci odbierają
- Dane są **trwałe** (zapisywane na dysk, można je odtworzyć)

**Jak działa (prosto):**
```
Sensor A → |             | → Dashboard 1
Sensor B → | Kafka Topic | → Dashboard 2
Sensor C → |             | → ML Model
```
Kafka = pośrednik, który **rozdziela** dane do wielu odbiorców.

**Kiedy używać:**
- Miliony eventów/sekundę
- Wiele systemów potrzebuje tych samych danych
- Potrzebujesz "replay" (odtworzenie starych eventów)
- Enterprise projects

**Przykłady:**
- Netflix (tracking użytkowników)
- Uber (lokalizacja kierowców)
- LinkedIn (activity streams)

**Poziom trudności:** 🔴🔴🔴 Trudny (wymaga clustra, Zookeeper, konfiguracji)

**Integracja z Dash:**
```python
from kafka import KafkaConsumer

consumer = KafkaConsumer('sensor-topic')
for message in consumer:
    # Zapisz do bazy / wyślij do Dash
    store_in_db(message.value)
```

---

### 2. WebSockets - "Dwukierunkowa Komunikacja"

**Co to jest:**
- Protokół sieciowy pozwalający na **dwukierunkową** komunikację
- Serwer może **sam wysłać** dane do klienta (nie czeka na request)
- Połączenie jest **trwałe** (nie zamyka się po każdym response)

**Jak działa (prosto):**
```
Klasyczny HTTP:
Klient:  "Daj dane" → Serwer: "OK, masz"
Klient:  "Daj dane" → Serwer: "OK, masz"
(każdy request osobno)

WebSocket:
Klient: "Połącz się"
Serwer: "OK, połączenie otwarte"
Serwer → Klient: "Nowe dane!"  (sam wysyła!)
Serwer → Klient: "Nowe dane!"
(jedno połączenie, dane płyną)
```

**Kiedy używać:**
- Komunikatory (chat, WhatsApp)
- Live notifications
- Gry online (multiplayer)
- Dashboardy z live update

**Przykłady:**
- Slack (chat)
- Trading platforms (live prices)
- Google Docs (collaborative editing)

**Poziom trudności:** 🟡🟡 Średni

**Integracja z Dash:**
Dash **nie wspiera natywnie** WebSockets, ale można użyć:
- `dash-extensions` (biblioteka community)
- Flask-SocketIO + Dash

```python
# Przykład z dash-extensions
from dash_extensions import WebSocket

app.layout = html.Div([
    WebSocket(url="ws://localhost:5000/data", id="ws"),
    dcc.Graph(id="graph")
])

@app.callback(Output("graph", "figure"), Input("ws", "message"))
def update(message):
    # message przychodzi z WebSocket
    return create_figure(message)
```

---

### 3. MQTT - "Protokół dla IoT"

**Co to jest:**
- Lekki protokół **publish/subscribe** dla urządzeń IoT
- Zaprojektowany dla **małych urządzeń** (czujniki, mikrokontrolery)
- Bardzo **niskie zużycie energii i łącza**

**Jak działa (prosto):**
```
Temperature Sensor → |              | → Dashboard
Humidity Sensor    → | MQTT Broker  | → Mobile App
Motion Sensor      → |              | → Alarm System
```
Broker = pośrednik (jak Kafka, ale lżejszy).

**Kiedy używać:**
- IoT (czujniki, smart home)
- Urządzenia z **ograniczonymi zasobami**
- Niestabilne połączenia (mobile networks)
- Publish-subscribe pattern

**Przykłady:**
- Smart home (czujniki temperatury, wilgotności)
- Przemysł 4.0 (fabryki, maszyny)
- Monitoring pojazdów (GPS trackers)

**Poziom trudności:** 🟢🟡 Łatwy-średni

**Popularne brokery:**
- Mosquitto (open-source, najpopularniejszy)
- HiveMQ
- AWS IoT Core

**Integracja z Dash:**
```python
import paho.mqtt.client as mqtt

def on_message(client, userdata, message):
    data = message.payload.decode()
    # Zapisz do bazy / aktualizuj Dash
    store_in_redis(data)

client = mqtt.Client()
client.connect("mqtt.broker.com")
client.subscribe("sensors/temperature")
client.on_message = on_message
client.loop_start()  # Nasłuchuj w tle
```

---

### 4. Redis Streams - "In-Memory Streaming"

**Co to jest:**
- **Redis** (baza in-memory) + funkcjonalność streamingu
- Szybkie (wszystko w RAM)
- Prostsze od Kafka, ale mniej funkcji

**Jak działa (prosto):**
```
Producer → Redis Stream → Consumer(s)
           (w pamięci)
```

**Kiedy używać:**
- **Średnie wolumeny** (nie miliony/sec jak Kafka)
- Potrzebujesz **szybkości** (in-memory)
- Już używasz Redis (cache, sessions)
- Prostsze od Kafka, ale real-time

**Przykłady:**
- Activity feeds (social media)
- Notifications
- Real-time analytics (moderate scale)

**Poziom trudności:** 🟡 Średni

**Integracja z Dash:**
```python
import redis

r = redis.Redis()

@app.callback(...)
def update(n):
    # Czytaj najnowsze z stream
    data = r.xread({'my-stream': '0-0'}, count=100)
    return create_figure(data)
```

---

### 5. Server-Sent Events (SSE) - "Prosty Streaming"

**Co to jest:**
- Prosty protokół HTTP pozwalający serwerowi **wysyłać** dane do klienta
- Jednokierunkowy (serwer → klient, nie odwrotnie)
- Wbudowany w przeglądarki (EventSource API)

**Jak działa (prosto):**
```
Klient otwiera połączenie
Serwer → Klient: "event 1"
Serwer → Klient: "event 2"
Serwer → Klient: "event 3"
(połączenie pozostaje otwarte)
```

**Kiedy używać:**
- Proste **jednoikierunkowe** streaming (serwer → klient)
- Live updates (notifications, feeds)
- Nie potrzebujesz wysyłać z klienta do serwera
- Chcesz **prostoty** (łatwiejsze niż WebSockets)

**Przykłady:**
- Live news feeds
- Stock tickers
- Notifications

**Poziom trudności:** 🟢 Łatwy

**Różnica vs WebSockets:**
- SSE: **Jednokierunkowe** (serwer → klient)
- WebSockets: **Dwukierunkowe** (serwer ↔ klient)

**Integracja z Dash:**
Dash nie wspiera natywnie, ale można:
```python
# Flask + SSE
from flask import Response

@server.route('/stream')
def stream():
    def event_stream():
        while True:
            data = get_new_data()
            yield f"data: {data}\n\n"
    return Response(event_stream(), mimetype="text/event-stream")
```

---

### 6. Apache Flink / Spark Streaming - "Big Data Streaming"

**Co to jest:**
- Framework do **przetwarzania** danych streaming
- Nie tylko przesyłają dane, ale je **analizują w locie**
- Dla **bardzo dużych** wolumenów (big data)

**Przykład:**
```
Milion eventów/sec → Flink → Agregacje → Dashboard
                      (avg, sum, count w locie)
```

**Kiedy używać:**
- **Big data** (terabajty/dzień)
- Potrzebujesz **przetwarzania w locie** (avg, sum, join)
- Kompleksowe analizy real-time
- Enterprise scale

**Poziom trudności:** 🔴🔴🔴 Bardzo trudny

**To NIE jest dla małych projektów!**

---

## 📊 Dash + Prawdziwe Źródła - Jak To Połączyć?

Dash **NIE zastępuje** Kafka/MQTT/WebSockets - Dash to tylko **front-end** (wizualizacja).

### Typowa architektura:

```
┌─────────────┐
│ Źródło      │ (sensory, API, logi)
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Streaming   │ (Kafka, MQTT, WebSockets)
│ Platform    │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Storage     │ (PostgreSQL, Redis, InfluxDB)
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Dash App    │ (czyta z bazy co X sekund)
└─────────────┘
```

### Wzorzec 1: Kafka → DB → Dash (najpopularniejszy)
```python
# Skrypt 1: Konsument Kafka (działa w tle)
from kafka import KafkaConsumer
import psycopg2

consumer = KafkaConsumer('sensor-data')
for message in consumer:
    # Zapisz do PostgreSQL
    save_to_db(message.value)

# Skrypt 2: Dash (odczytuje z DB)
dcc.Interval(interval=2000)  # Co 2 sekundy

@app.callback(...)
def update(n):
    df = pd.read_sql('SELECT * FROM sensors ORDER BY timestamp DESC LIMIT 1000', conn)
    return px.line(df, x='timestamp', y='value')
```

**Dlaczego tak?**
- Kafka obsługuje miliony eventów
- Baza to "buffer" - Dash nie musi nadążać za strumieniem
- Dane są trwałe (możesz pokazać historię)

---

### Wzorzec 2: MQTT → Redis → Dash (szybki, in-memory)
```python
# Skrypt 1: Subscriber MQTT
import paho.mqtt.client as mqtt
import redis

r = redis.Redis()

def on_message(client, userdata, message):
    # Zapisz do Redis (szybkie!)
    r.lpush('sensor-queue', message.payload)
    r.ltrim('sensor-queue', 0, 999)  # Trzymaj ostatnie 1000

# Skrypt 2: Dash
@app.callback(...)
def update(n):
    data = r.lrange('sensor-queue', 0, -1)
    return create_figure(data)
```

---

### Wzorzec 3: WebSocket → Dash (zaawansowany)
```python
# Wymaga dash-extensions lub Flask-SocketIO
from dash_extensions import WebSocket

app.layout = html.Div([
    WebSocket(url="ws://data-server.com/stream", id="ws"),
    dcc.Graph(id="graph")
])

@app.callback(Output("graph", "figure"), Input("ws", "message"))
def update(ws_data):
    # ws_data przychodzi na bieżąco z WebSocket
    return create_figure(json.loads(ws_data))
```

---

## 🌐 Deployment i Publikacja - Jak Udostępnić?

Masz dashboard - jak go **opublikować** żeby inni mogli używać?

### Opcja 1: Lokalne Uruchomienie (tylko Ty)
```bash
python app.py  # http://127.0.0.1:8050/
```
✅ Darmowe, proste  
❌ Tylko na Twoim komputerze

---

### Opcja 2: Cloud Hosting - Render, Railway (darmowy tier)
**Jak to działa:**
1. Push kod na GitHub
2. Połącz Render z repo
3. Render automatycznie deployuje
4. Dostajesz publiczny URL: `https://my-dashboard.onrender.com`

✅ Darmowy tier (do 15 min sleep)  
✅ Prosty deployment  
✅ Publiczny dostęp  
❌ Sleep po 15 min nieaktywności (free tier)

**Kroki:**
```bash
# 1. requirements.txt
dash
plotly
pandas

# 2. Push na GitHub
git init
git add .
git commit -m "Initial commit"
git push

# 3. Render.com → New Web Service → Connect GitHub repo
```

---

### Opcja 3: Plotly Chart Studio (dla prostych wykresów)
**Jak to działa:**
```python
import plotly.express as px
import chart_studio.plotly as py

fig = px.line(df, x='date', y='value')
py.plot(fig, filename='my-chart', auto_open=True)
# Dostajesz URL: https://chart-studio.plotly.com/~username/123
```

✅ Bardzo proste  
✅ Hosting Plotly  
❌ **Płatne** dla prywatnych wykresów  
❌ Tylko wykresy, nie pełen dashboard

---

### Opcja 4: Własny Serwer (VPS, AWS, Azure)
**Dla zaawansowanych:**
- Wynajmujesz serwer (AWS EC2, DigitalOcean Droplet)
- Instalujesz Python, Dash, bazy danych
- Konfigurujesz nginx, SSL, domeny
- **Pełna kontrola**, ale wymaga DevOps skills

✅ Pełna kontrola  
✅ Bez limitów  
❌ Drogie ($5-100+/miesiąc)  
❌ Wymaga umiejętności DevOps

---

### Opcja 5: Dash Enterprise (komercyjne)
Oficjalna platforma Plotly do deploymentu Dash apps.

✅ Profesjonalne (auth, scaling, monitoring)  
❌ **Bardzo drogie** (tysiące $/rok)

**To dla korporacji, nie dla małych projektów.**

---

## 🎯 Podsumowanie - Kiedy Co Wybrać?

### Flowchart - Wybór Rozwiązania:

```
START: Potrzebuję wizualizacji live data
  │
  ├─ To tylko prototyp/demo?
  │   └─ TAK → Fake Live (dcc.Interval + random)
  │
  ├─ Aktualizacja co 5+ sekund wystarczy?
  │   └─ TAK → Polling DB (dcc.Interval + SQL)
  │
  ├─ Potrzebuję real-time (< 1 sec)?
  │   │
  │   ├─ Małe/średnie wolumeny?
  │   │   └─ Redis Streams / WebSockets / SSE
  │   │
  │   ├─ IoT (czujniki)?
  │   │   └─ MQTT
  │   │
  │   ├─ Duże wolumeny (tysiące events/sec)?
  │   │   └─ Kafka
  │   │
  │   └─ Big Data (miliony events/sec)?
  │       └─ Kafka + Flink/Spark
```

---

### Tabela Porównawcza:

| Rozwiązanie | Opóźnienie | Trudność | Koszt | Kiedy używać |
|-------------|------------|----------|-------|-------------|
| **Fake Live** (random) | N/A | 🟢 Łatwe | Darmowe | Prototypy, nauka |
| **Polling DB** | 5-60 sec | 🟢 Łatwe | Niski | Małe/średnie projekty |
| **Redis Streams** | < 1 sec | 🟡 Średnie | Średni | Średnie wolumeny |
| **WebSockets** | < 1 sec | 🟡 Średnie | Niski | Dwukierunkowa komunikacja |
| **SSE** | < 1 sec | 🟢 Łatwe | Niski | Jednokierunkowy streaming |
| **MQTT** | < 1 sec | 🟡 Średnie | Niski | IoT, czujniki |
| **Kafka** | < 100ms | 🔴 Trudne | Wysoki | Duże wolumeny, enterprise |
| **Flink/Spark** | < 100ms | 🔴🔴 Bardzo trudne | Bardzo wysoki | Big Data |

---

## 📚 Dodatkowe Zasoby - Jeśli Chcesz Zgłębić

### Kafka:
- Dokumentacja: https://kafka.apache.org/
- Tutorial: Confluent Kafka Python

### MQTT:
- Mosquitto (broker): https://mosquitto.org/
- Python client: `paho-mqtt`

### WebSockets:
- `dash-extensions` (WebSocket dla Dash)
- Flask-SocketIO

### Redis Streams:
- Dokumentacja: https://redis.io/topics/streams-intro
- Python: `redis-py`

### Deployment:
- Render: https://render.com/
- Railway: https://railway.app/
- Heroku (płatny): https://heroku.com/

---

## ❓ FAQ - Najczęstsze Pytania

**Q: Czy Dash jest wystarczający do live data?**  
A: Dash to **front-end** - pokazuje dane. Potrzebujesz **źródła** danych (Kafka, MQTT, DB). Dash + Polling DB to **90% przypadków**.

**Q: Czy muszę znać Kafka żeby robić dashboardy?**  
A: **NIE!** Większość projektów wystarcza Polling DB (co 5-30 sekund). Kafka tylko dla enterprise.

**Q: Jak udostępnić dashboard kolegom?**  
A: Render.com (darmowy tier) - push na GitHub, connect, dostaniesz URL.

**Q: Co wybrać dla IoT (czujniki Arduino)?**  
A: **MQTT** - zaprojektowany dla IoT. Lekki, niskie zużycie energii.

**Q: Czy mogę użyć Dash do trading (giełda)?**  
A: Technicznie tak, ale profesjonalne platformy używają **WebSockets** (ms opóźnienia). Dash Polling = zbyt wolne.

**Q: Czy Redis Streams to zamiennik Kafka?**  
A: Dla **średnich** wolumenów - TAK. Dla milionów events/sec - Kafka lepszy.

---

## 🎓 Wnioski

1. **"Live data" ma wiele poziomów** - od fake (random) po enterprise (Kafka)
2. **90% projektów = Polling DB** - proste i wystarczające
3. **Dash to front-end** - potrzebujesz źródła danych (Kafka, MQTT, DB)
4. **Prawdziwy streaming = złożona infrastruktura** - tylko dla dużych projektów
5. **Start small** - zacznij od Polling DB, skaluj gdy trzeba

**Nie musisz znać Kafka/MQTT żeby robić świetne dashboardy!**  
**Ale dobrze wiedzieć że istnieją - gdy projekt urośnie.** 🚀